# Load CMS 2026 HSD Reference File to BigQuery
**Source:** CMS 2026 HSD Reference File (12-17-2025)  
**Target tables:**
- `A870800_medicare_supply_demand_ref_hsd_provider_min`
- `A870800_medicare_supply_demand_ref_hsd_facility_min`

In [ ]:
import pandas as pd
import numpy as np
from google.cloud import bigquery

PROJECT   = 'anbc-hcb-dev'
DATASET   = 'provider_ds_netconf_data_hcb_dev'
HSD_FILE  = 'ma_reference_file_12-17-2025.xlsx'  # update path as needed
STATE     = 'FL'

## 1. Load Provider Minimums Sheet

In [ ]:
raw_prov = pd.read_excel(HSD_FILE, sheet_name='Minimum Provider #s', header=None)

# row 1 = headers, row 2 = specialty codes (skip), row 3+ = data
raw_prov.columns = raw_prov.iloc[1].tolist()
raw_prov = raw_prov.iloc[3:].reset_index(drop=True)

# filter florida
prov = raw_prov[raw_prov['ST'] == STATE].copy()
print(f'Provider rows: {len(prov)}')
prov.head(3)

## 2. Clean Provider Sheet

In [ ]:
# clean column names
prov.columns = (
    prov.columns
    .str.strip()
    .str.replace(r'\n', ' ', regex=True)
    .str.replace(r'\s+', ' ', regex=True)
    .str.replace(r'[^a-zA-Z0-9_ ]', '', regex=True)
    .str.strip()
    .str.replace(' ', '_')
    .str.upper()
)

# drop sub-component primary care cols (not needed for analysis)
drop_cols = [
    'GENERAL_PRACTICE_SEE_NOTES',
    'FAMILY_PRACTICE_SEE_NOTES',
    'INTERNAL_MEDICINE_SEE_NOTES',
    'GERIATRICS_SEE_NOTES',
    'PRIMARY_CARE__PHYSICIAN_ASSISTANTS_SEE_NOTES',
    'PRIMARY_CARE__NURSE_PRACTITIONERS_SEE_NOTES',
]
prov.drop(columns=[c for c in drop_cols if c in prov.columns], inplace=True)

# cast types
prov['TOTAL_BENEFICIARIES']                 = pd.to_numeric(prov['TOTAL_BENEFICIARIES'], errors='coerce').astype('Int64')
prov['95TH_PERCENTILE_BASE_POPULATION_RATIO'] = pd.to_numeric(prov['95TH_PERCENTILE_BASE_POPULATION_RATIO'], errors='coerce').astype(float)
prov['BENEFICIARIES_REQUIRED_TO_COVER']     = pd.to_numeric(prov['BENEFICIARIES_REQUIRED_TO_COVER'], errors='coerce').astype('Int64')

# all specialty count columns to Int64
skip_cols = {'COUNTY', 'ST', 'COUNTY_STATE', 'SSACD', 'COUNTY_DESIGNATION',
             'TOTAL_BENEFICIARIES', '95TH_PERCENTILE_BASE_POPULATION_RATIO',
             'BENEFICIARIES_REQUIRED_TO_COVER'}
for col in prov.columns:
    if col not in skip_cols:
        prov[col] = pd.to_numeric(prov[col], errors='coerce').astype('Int64')

print(prov.dtypes)
prov.head(3)

## 3. Load Facility Minimums Sheet

In [ ]:
raw_fac = pd.read_excel(HSD_FILE, sheet_name='Minimum Facility #s', header=None)

raw_fac.columns = raw_fac.iloc[1].tolist()
raw_fac = raw_fac.iloc[3:].reset_index(drop=True)

fac = raw_fac[raw_fac['ST'] == STATE].copy()
print(f'Facility rows: {len(fac)}')
fac.head(3)

## 4. Clean Facility Sheet

In [ ]:
fac.columns = (
    fac.columns
    .str.strip()
    .str.replace(r'\n', ' ', regex=True)
    .str.replace(r'\s+', ' ', regex=True)
    .str.replace(r'[^a-zA-Z0-9_ ]', '', regex=True)
    .str.strip()
    .str.replace(' ', '_')
    .str.upper()
)

fac['TOTAL_BENEFICIARIES']                   = pd.to_numeric(fac['TOTAL_BENEFICIARIES'], errors='coerce').astype('Int64')
fac['95TH_PERCENTILE_BASE_POPULATION_RATIO'] = pd.to_numeric(fac['95TH_PERCENTILE_BASE_POPULATION_RATIO'], errors='coerce').astype(float)
fac['BENEFICIARIES_REQUIRED_TO_COVER']       = pd.to_numeric(fac['BENEFICIARIES_REQUIRED_TO_COVER'], errors='coerce').astype('Int64')

skip_cols = {'COUNTY', 'ST', 'COUNTY_STATE', 'SSACD', 'COUNTY_DESIGNATION',
             'TOTAL_BENEFICIARIES', '95TH_PERCENTILE_BASE_POPULATION_RATIO',
             'BENEFICIARIES_REQUIRED_TO_COVER'}
for col in fac.columns:
    if col not in skip_cols:
        fac[col] = pd.to_numeric(fac[col], errors='coerce').astype('Int64')

print(fac.dtypes)
fac.head(3)

## 5. Export to CSV (optional validation step)

In [ ]:
prov.to_csv('hsd_provider_min_fl.csv', index=False)
fac.to_csv('hsd_facility_min_fl.csv', index=False)
print('CSVs written')

## 6. Define BigQuery Schemas

In [ ]:
client = bigquery.Client(project=PROJECT)

def bq_schema_from_df(df):
    type_map = {
        'object':  bigquery.enums.SqlTypeNames.STRING,
        'int64':   bigquery.enums.SqlTypeNames.INT64,
        'Int64':   bigquery.enums.SqlTypeNames.INT64,
        'float64': bigquery.enums.SqlTypeNames.FLOAT64,
        'bool':    bigquery.enums.SqlTypeNames.BOOL,
    }
    schema = []
    for col, dtype in df.dtypes.items():
        bq_type = type_map.get(str(dtype), bigquery.enums.SqlTypeNames.STRING)
        schema.append(bigquery.SchemaField(col, bq_type))
    return schema

prov_schema = bq_schema_from_df(prov)
fac_schema  = bq_schema_from_df(fac)

print('Provider schema:')
for f in prov_schema:
    print(f'  {f.name}: {f.field_type}')

## 7. Load to BigQuery

In [ ]:
def load_to_bq(df, table_name, schema):
    table_id = f'{PROJECT}.{DATASET}.{table_name}'
    job_config = bigquery.LoadJobConfig(
        schema=schema,
        write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
    )
    job = client.load_table_from_dataframe(df, table_id, job_config=job_config)
    job.result()
    print(f'Loaded {len(df)} rows to {table_id}')

load_to_bq(
    prov,
    'A870800_medicare_supply_demand_ref_hsd_provider_min',
    prov_schema
)

load_to_bq(
    fac,
    'A870800_medicare_supply_demand_ref_hsd_facility_min',
    fac_schema
)

## 8. Validate

In [ ]:
for tbl in [
    'A870800_medicare_supply_demand_ref_hsd_provider_min',
    'A870800_medicare_supply_demand_ref_hsd_facility_min'
]:
    q = f'SELECT COUNT(*) as cnt FROM `{PROJECT}.{DATASET}.{tbl}`'
    result = client.query(q).result()
    for row in result:
        print(f'{tbl}: {row.cnt} rows')